<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem850_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
class SegmentTree:
    def __init__(self, y_coordinates):
        self.y_coords = y_coordinates
        self.num_intervals = len(y_coordinates) - 1

        # Standard 4 * N tracking arrays
        # cover_count[node] answers:
        # How many active rectangles are directly covering this whole segment?
        self.cover_count = [0] * (4 * self.num_intervals)

        # total_length[node] answers:
        # How much y-length inside this segment is currently covered.
        #
        # This is NOT the physical size of the segment itself.
        # It is the union length of active rectangles in this segment.
        self.total_length = [0] * (4 * self.num_intervals)


    def update(self, query_low, query_high, active_delta):
        # Add or remove a y-interval segment from the tree
        self._update_node(1, 0, self.num_intervals, query_low, query_high, active_delta)


    def get_current_active_length(self):
        # Returns the total y-length currently covered by active rectangles
        return self.total_length[1]


    def _update_node(self, node, segment_low, segment_high, query_low, query_high, delta):
        # No overlap whatsoever
        if query_high <= segment_low or segment_high <= query_low:
            return

        # Complete overlap: this segment tree node is fully covered by the query
        # => The rectangle completely covers this entire node interval
        if query_low <= segment_low and segment_high <= query_high:
            self.cover_count[node] += delta

        # Partial overlap: recurse down to left and right children
        else:
            mid = (segment_low + segment_high) // 2
            self._update_node(node * 2, segment_low, mid, query_low, query_high, delta)
            self._update_node(node * 2 + 1, mid, segment_high, query_low, query_high, delta)

        # Pull-up / Maintain the length properties for this node
        # Steps:
        # 1. Update children (if needed)
        # 2. Children know their lengths
        # 3. Parent combines them
        #
        # cover_count[node] > 0: this node is completely covered by something
        if self.cover_count[node] > 0:
            # If completely covered by at least one active rectangle, use the actual physical distance
            self.total_length[node] = self.y_coords[segment_high] - self.y_coords[segment_low]

        # cover_count[node] == 0:
        # This node is not fully covered by any active rectangle,
        # so the answer comes from its children.
        else:
            # If not fully covered, it depends entirely on its children
            if segment_high - segment_low == 1:
                # Base case: it's a leaf interval and nothing covers it
                self.total_length[node] = 0

            else:
                # Internal node: sum up the lengths of both child segments
                self.total_length[node] = self.total_length[node * 2] + self.total_length[node * 2 + 1]


class Solution:
    def rectangleArea(self, rectangles):
        MOD = 10**9 + 7
        total_area = 0

        # Parse geometric boundaries into unique events and tracking points
        sweep_events = []
        y_points = set()

        for x1, y1, x2, y2 in rectangles:
            # Rectangle enters the sweep line
            sweep_events.append((x1, y1, y2, 1))

            # Rectangle leaves the sweep line
            sweep_events.append((x2, y1, y2, -1))

            y_points.add(y1)
            y_points.add(y2)

        # Coordinate compression for y-axis to map massive coordinate values to dense indices
        sorted_y_coords = sorted(y_points)
        y_to_index = {y_val: idx for idx, y_val in enumerate(sorted_y_coords)}

        # Sort the vertical sweep line events by their x-coordinate moving left-to-right
        sweep_events.sort()

        # Initialize the segment tree with our compressed layout
        tree = SegmentTree(sorted_y_coords)

        previous_x = sweep_events[0][0]

        # Execute the sweep line loop
        # The execution order:
        # 1. Calculate the distance we already traveled
        # 2. Use the old active rectangles
        # 3. Apply event changes
        # 4. Move forward
        for current_x, y1, y2, active_delta in sweep_events:
            # Calculate the width of the vertical slice we just passed through
            slice_width = current_x - previous_x

            # Area of this slice = width * total height currently tracked by the segment tree
            total_area += slice_width * tree.get_current_active_length()
            total_area %= MOD

            # Update the segment tree state by inserting or removing the current interval
            tree.update(
                query_low=y_to_index[y1],
                query_high=y_to_index[y2],
                active_delta=active_delta
            )

            previous_x = current_x

        return total_area % MOD


In [14]:
solution = Solution()

In [15]:
rectangles = [[0,0,2,2],[1,0,2,3],[1,0,3,1]]

print(solution.rectangleArea(rectangles))

6


In [16]:
rectangles = [[0,0,1000000000,1000000000]]

print(solution.rectangleArea(rectangles))

49
